# Assign cell types and create pseudobulk

+ Input: 'adata_clusters.h5ad'
+ Output: '{cell_type}_pseudobulk.tsv'
+ Next step: Subclustering, TensorQTL processing 


In [1]:
# This cell is labelled 'parameters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

Processing plate: plate1


### Set Variables

In [1]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from scanpy_init_env import *
from scanpy_utils import *
from scanpy_gene_lists import *

### Load Data

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = sc.read(scanpy_dir + f'adata_clusters_v3.h5ad')

### Pull out raw counts 

In [ ]:
logger.info("Restoring raw counts ...")
adata.layers["counts"] = adata.X.copy()
print(adata.layers)
print(adata.layers["counts"])

# Sanity check
adata.X[:10, :10].todense()

### Find overlapping samples with genotype data

In [ ]:
logger.info("Load  ...")
geno_sample_file = Path(root_dir + 'results/04GENOTYPES-POST/filtered/geno_sample_list.txt')
if not geno_sample_file.exists():
    print("Overlapping samples file does not exist")
else:
    print("Overlapping samples file already exists")
    with open(geno_sample_file, "r") as f:
        sample_set = set(f.read().splitlines())


### Assign cell types

In [ ]:
logger.info("Setting cluster IDs ...")
# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.2'].map(cluster_anns)

# Check the new column
print(adata.obs[['leiden_harmony_0.2', 'cell_type']].head())

# Manually generate Pseudobulk

In [ ]:
logger.info("Generating psudobulk ...")
pseudblk_dfs = {}

for cell_type in adata.obs.cell_type.unique():
    print(f"Processing cell type: {cell_type}")
    
    # Subset the data for the current cell type (create a copy)
    cell_subset = adata[adata.obs['cell_type'] == cell_type].copy()
    
    # Initialize an empty list to store pseudobulk objects
    combined_pseudobulk_lst = []

    # Loop through unique samples
    for sample in cell_subset.obs['sample'].unique():

        # Remove samples not in the overlapping set
        if sample not in sample_set:  
            continue

        samp_cell_subset = cell_subset[cell_subset.obs['sample'] == sample].copy()

        # Extract the raw counts
        counts_matrix = samp_cell_subset.X
        
        # Ensure counts are stored as integers
        print(f"Max count value (should be integer): {counts_matrix.max()}")
        
        # Sum counts across cells in the sample
        pseudobulk_counts = counts_matrix.sum(axis=0).A1 if hasattr(counts_matrix, "A1") else counts_matrix.sum(axis=0)
        
        # Create an AnnData object with summed counts
        samp_adata = sc.AnnData(X=pseudobulk_counts.reshape(1, -1), var=samp_cell_subset.var)
        samp_adata.obs_names = [sample]

        # Append to the list
        combined_pseudobulk_lst.append(samp_adata)

    # Combine all samples into one AnnData object
    pseudobulk_adata = sc.concat(combined_pseudobulk_lst, join="outer")

    # Convert pseudobulk_adata to a DataFrame for the current cell type
    pseudobulk_df = pd.DataFrame(pseudobulk_adata.X, columns=pseudobulk_adata.var_names, index=pseudobulk_adata.obs_names)
    print(f"pseudobulk_df dims: {pseudobulk_df.shape}")

    # Save the DataFrame to the dictionary
    pseudblk_dfs[cell_type] = pseudobulk_df

    # Save the pseudobulk data for this cell type to a CSV file
    pseudobulk_df.to_csv(os.path.join(scanpy_dir, f'pseudobulk/{cell_type}_pseudobulk.csv'))

In [ ]:
for df in pseudblk_dfs:
    print(pseudblk_dfs[df].shape)